# Phase 2 — Treatment and Outcome Audit

Phase 1 is complete and frozen. This notebook audits treatment variables and the current outcome definition without performing causal modeling.

The goal is to determine whether the existing treatment variables can support a defensible treatment-specific time zero and whether the outcome definition is suitable for later causal work.


In [ ]:
import json
from pathlib import Path

import pandas as pd

DATA_DIR = Path(r'c:/Users/anura/OneDrive/Desktop/capstone2/master')
cols = ['study_id', 'SubjectID', 'hadm_id', 'ecg_time', 'Label_Troponin', 'is_MI', 'index_mi_time', 'time_diff_hours', 'statin_before', 'statin_during', 'statin_after', 'beta_blocker_before', 'beta_blocker_during', 'acei_before', 'acei_during', 'aspirin_before', 'aspirin_during', 'vasopressor_during', 'mech_vent_during']
files = [DATA_DIR / 'train.parquet', DATA_DIR / 'validation.parquet', DATA_DIR / 'test.parquet']
df = pd.concat([pd.read_parquet(p, columns=cols) for p in files], ignore_index=True)
print('combined_rows', len(df))
print('split_rows', {split: len(pd.read_parquet(DATA_DIR / f'{split}.parquet')) for split in ['train','validation','test']})
print('treatment_columns', [c for c in df.columns if any(k in c.lower() for k in ['statin','beta','acei','aspirin','vasopressor','vent'])])


## 1. Use the actual three-way dataset

The verified split files are used directly: train = 262,813 rows, validation = 56,319 rows, test = 56,798 rows, combined = 375,930 rows.

The main treatment audit uses the combined dataset to describe prevalence and temporal structure, while split-specific checks are preserved and reported separately when needed.


## 2. Treatment variables and underlying meaning

The Phase F dataset schema states that the treatment columns are windowed indicators, not exact medication timestamps. In particular:

- `*_before` indicates pre-admission / pre-encounter medication use
- `*_during` indicates in-hospital use during the admission
- `statin_after` indicates post-discharge use after discharge

This means the parquet contains binary treatment indicators tied to encounter windows, not drug administration timestamps. The exact administration times are not available in the current dataset.


## 3. Treatment prevalence

The next code cells quantify exposure prevalence at ECG-row and admission levels.


In [ ]:
cols = ['statin_before','statin_during','statin_after','beta_blocker_before','beta_blocker_during','acei_before','acei_during','aspirin_before','aspirin_during','vasopressor_during','mech_vent_during']
rows = {c: {'exposed_count': int(df[c].eq(1).sum()), 'unexposed_count': int(df[c].eq(0).sum()), 'missing_count': int(df[c].isna().sum()), 'exposure_percentage': round(float(df[c].eq(1).mean() * 100), 3)} for c in cols}
adm = df.groupby('hadm_id')[cols].max().reset_index()
admissions = {c: {'exposed_count': int(adm[c].fillna(0).gt(0).sum()), 'unexposed_count': int(adm[c].fillna(0).eq(0).sum()), 'missing_count': int(adm[c].isna().sum()), 'exposure_percentage': round(float(adm[c].fillna(0).gt(0).mean() * 100), 3)} for c in cols}
print(json.dumps({'ecg_row_level': rows, 'admission_level': admissions}, indent=2))


## 4. Treatment consistency within admission

Because Phase 1 established admission-level as the primary unit, binary treatment values should be consistent within `hadm_id` when they describe an admission-level treatment flag. The next code checks this.


In [ ]:
consistency = {}
for c in cols:
    vals = df.groupby('hadm_id')[c].agg(lambda x: set(pd.Series(x).dropna().unique().tolist()))
    bad = int(vals.apply(lambda v: len(v) > 1).sum())
    consistency[c] = {'n_admissions_with_inconsistent_values': bad, 'percentage_of_admissions_affected': round(100 * bad / len(vals), 3) if len(vals) else 0.0}
print(json.dumps(consistency, indent=2))


## Temporal interpretation correction

`time_diff_hours` is not a universal ECG-to-MI variable. For MI patients it is `ecg_time - index_mi_time`; for `Control_Symptomatic` rows it is based on the admission-time anchor/fallback and is not relative to an MI event. Use `Label_Troponin` for the established MI temporal cohort labels. The master modeling dataset contains zero `MI_Post-Incident` rows because that category was excluded during dataset construction. Do not classify control rows as post-MI from a positive `time_diff_hours`, and do not use this field alone to establish whether a covariate preceded the treatment-specific causal baseline/index ECG.


## 5. Outcome audit

The categorical cohort label and raw temporal variable are distinct. `Label_Troponin` carries the established cohort categories; `time_diff_hours` is a raw field whose reference differs between MI patients and controls. The master dataset contains no `MI_Post-Incident` rows: that category was intentionally excluded during modeling-data construction.


In [ ]:
print('is_MI_is_binary', bool(df['is_MI'].nunique() <= 2))
print('Label_Troponin_counts', df['Label_Troponin'].value_counts(dropna=False).to_dict())
print('is_MI_by_Label_Troponin')
print(pd.crosstab(df['Label_Troponin'], df['is_MI'], dropna=False).to_string())
print('label-specific index_mi_time/time_diff_hours availability')
print(df.groupby('Label_Troponin', dropna=False).agg(rows=('study_id','size'), index_mi_time_nonnull=('index_mi_time','count'), time_diff_hours_nonnull=('time_diff_hours','count')).to_string())
print('time_diff_hours is not a universal ECG-to-MI variable; do not use it to infer MI ordering for controls.')
print('outcome_definition_assessment', 'UNRESOLVED — requires later causal outcome definition/audit.')


## 6. Temporal ordering and baseline eligibility

Exact medication timestamps are unavailable, so the ordering of treatment assignment relative to ECG and MI cannot be established from the treatment flags. `time_diff_hours` does not establish whether variables were measured before a treatment-specific causal baseline/index ECG. It is only an ECG-to-index-MI difference for MI patients; for controls it uses an admission-time anchor/fallback. The established categorical MI temporal labels remain `Label_Troponin`, not classifications inferred from the sign of this field.


In [ ]:
mi_labels = df['Label_Troponin'].isin(['MI_Pre-Incident', 'MI_Acute_Presentation'])
mi_times = df.loc[mi_labels].copy()
both_times = mi_times['ecg_time'].notna() & mi_times['index_mi_time'].notna()
print('MI_Post-Incident rows', int(df['Label_Troponin'].eq('MI_Post-Incident').sum()))
print('MI-label rows with both ECG and index MI timestamps', int(both_times.sum()))
print('time_diff_hours is retained as a raw variable and is not used here to reclassify Label_Troponin.')
print('Treatment orderings: UNRESOLVED because exact treatment assignment/admin timestamps are unavailable.')


## Decision Record

Primary candidate treatment(s):
- UNRESOLVED. The available data do not provide exact treatment assignment timing for a defensible ECG-based baseline treatment.

Reason:
- The treatment variables are binary window indicators (`*_before`, `*_during`, `*_after`) defined relative to admission/discharge windows, not exact medication administration timestamps.

Treatments excluded from primary analysis:
- vasopressor_during, mech_vent_during are excluded from primary exposure modeling because they are acute supportive interventions rather than stable baseline treatments.

Treatment timing:
What is known:
- `*_before` variables are pre-admission baseline indicators.
- `*_during` variables are in-hospital, acute-treatment indicators.
- `statin_after` is post-discharge and therefore downstream relative to the MI episode.

What is unknown:
- Exact medication start or administration time relative to ECG and MI.
- Whether a binary `*_before` flag reflects treatment assigned at the clinically relevant baseline time rather than chronic pre-existing therapy.

Outcome definition:
Current definition:
- `is_MI` is a binary MI indicator derived from `Label_Troponin` and the project MI labeling workflow.
- `Label_Troponin` is the established categorical MI temporal cohort label. The master modeling dataset has zero `MI_Post-Incident` rows because that category was intentionally excluded during dataset construction.
- `index_mi_time` availability is reported from the observed data by label; it is not assumed to be missing for all controls.
- `time_diff_hours` is ECG minus `index_mi_time` for MI patients. For `Control_Symptomatic` rows it uses the admission-time anchor/fallback and does not represent time relative to MI.

Outcome timing:
Assessment:
- Do not classify master-dataset rows as post-MI merely because `time_diff_hours > 0`; do not use that field alone to assess whether a covariate precedes the treatment-specific baseline/index ECG. The causal outcome definition remains unresolved.

Temporal ordering:
- Exact treatment-to-ECG-to-MI ordering is UNRESOLVED because treatment timestamps are unavailable. For controls, `time_diff_hours` is not MI-relative.

Causal time zero:
Status:
- UNRESOLVED for all candidate treatments because exact treatment-assignment timing is unavailable.

Treatment-specific time zero:
- UNRESOLVED for aspirin, statin, beta blocker, and ACE inhibitor.

Potential baseline treatments:
- statin_before, beta_blocker_before, acei_before, aspirin_before — only as chronic/pre-admission exposure indicators, not as exact baseline treatment assignment.

Potential acute treatments:
- statin_during, beta_blocker_during, acei_during, aspirin_during — likely acute or in-hospital treatment exposure, but they are not exact assignment indicators.

Potential downstream treatments:
- statin_after

Positivity concerns:
- Potentially important if some treatment strata are sparse or nearly all treated; requires later design-specific assessment.

Confounding-by-indication concerns:
- High. Medication use is likely correlated with disease severity and clinical indication, and the current data do not separate pre-existing treatment from treatment triggered by the acute episode.

Temporal inconsistencies:
- No within-admission treatment-flag inconsistencies were observed. This does not supply medication timestamps or resolve treatment ordering. Do not interpret `time_diff_hours` as a universal ECG-to-MI variable.

Unresolved issues:
- Exact treatment assignment timing relative to ECG and MI.
- Whether the binary `*_before` flag can serve as a valid baseline treatment for causal analysis.
- Whether the current MI label/time definition is sufficiently precise for future causal outcome analysis.

Phase 3 handoff:
- Phase 3 should only proceed after a separate causal outcome specification and a treatment-timing source audit is completed. The current Phase 2 result is that a defensible causal treatment time zero is not established from the existing parquet data.
